[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/21_gradient_clipping.ipynb)

# 🟢 简单：梯度范数裁剪

实现 **梯度范数裁剪** — 一种训练稳定性技术。

### 函数签名
```python
def clip_grad_norm(parameters, max_norm: float) -> float:
    # 原地裁剪梯度，使总范数 ≤ max_norm
    # 返回原始（未裁剪）总范数
```

### 算法
1. 计算总范数：`sqrt(sum(p.grad.norm()^2 for p in parameters))`
2. 如果总范数 > max_norm：将所有梯度按比例缩放 `max_norm / total`
3. 返回原始总范数

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ 在此实现你的代码

def clip_grad_norm(parameters, max_norm):
    pass  # 计算总范数，如有需要则裁剪，返回原始范数

# 梯度范数裁剪公式

## 1. 总梯度范数计算

首先计算所有参数梯度的总范数（L2范数）：

$$\text{total\_norm} = \sqrt{\sum_{p \in \text{parameters}} \| \nabla p \|_2^2}$$

其中：
- $\nabla p$ 表示参数 $p$ 的梯度张量
- $\| \nabla p \|_2$ 是梯度张量的L2范数（Frobenius范数）

对于单个张量 $\nabla p$，其L2范数为：

$$\| \nabla p \|_2 = \sqrt{\sum_{i} (\nabla p_i)^2}$$

所以总范数也可以写为所有梯度元素的平方和的平方根：

$$\text{total\_norm} = \sqrt{\sum_{p} \sum_{i} (\nabla p_i)^2}$$

## 2. 裁剪条件

比较总范数与阈值 $\text{max\_norm}$：

- 如果 $\text{total\_norm} \leq \text{max\_norm}$：不进行裁剪
- 如果 $\text{total\_norm} > \text{max\_norm}$：进行裁剪

## 3. 裁剪操作（缩放）

当需要进行裁剪时，计算缩放因子：

$$\text{clip\_coef} = \frac{\text{max\_norm}}{\text{total\_norm}}$$

然后对所有参数的梯度进行原地缩放：

$$\nabla p \leftarrow \nabla p \times \text{clip\_coef}$$

## 4. 完整的数学表示

综合起来，梯度范数裁剪可以写为：

$$\nabla p \leftarrow \begin{cases}
\nabla p & \text{if } \| \mathbf{g} \|_2 \leq \text{max\_norm} \\
\nabla p \times \frac{\text{max\_norm}}{\| \mathbf{g} \|_2} & \text{if } \| \mathbf{g} \|_2 > \text{max\_norm}
\end{cases}$$

其中 $\mathbf{g} = (\nabla p_1, \nabla p_2, \ldots, \nabla p_n)$ 是所有参数梯度的拼接向量，$\| \mathbf{g} \|_2$ 是其总L2范数。

等价地，可以统一写成：

$$\nabla p \leftarrow \nabla p \times \min\left(1, \frac{\text{max\_norm}}{\| \mathbf{g} \|_2}\right)$$



In [ ]:
def clip_grad_norm(parameters, max_norm: float) -> float:
    """
    原地裁剪梯度，使总范数 ≤ max_norm
    
    Args:
        parameters: 可迭代的参数张量（或单个张量）
        max_norm: 梯度总范数的最大允许值
    
    Returns:
        float: 原始（未裁剪）总范数
    """
    # 确保 parameters 是可迭代的
    if isinstance(parameters, torch.Tensor):
        parameters = [parameters]
    
    # 收集所有参数的梯度平方和
    total_norm_sq = 0.0
    for p in parameters:
        if p.grad is not None:
            # 计算每个权重张量的梯度张量的范数平方
            grad_norm_sq = p.grad.data.norm(2).item() ** 2
            total_norm_sq += grad_norm_sq
    
    # 计算总范数
    total_norm = total_norm_sq ** 0.5
    
    # 如果总范数超过阈值，进行裁剪
    if total_norm > max_norm:
        # 计算缩放因子
        clip_coef = max_norm / (total_norm + 1e-6)  # 添加小值防止除零
        
        # 对所有梯度进行原地缩放
        for p in parameters:
            if p.grad is not None:
                p.grad.data.mul_(clip_coef)
    
    return total_norm

In [ ]:
# 🧪 调试
p = torch.randn(100, requires_grad=True)
(p * 10).sum().backward()
print('裁剪前:', p.grad.norm().item())
orig = clip_grad_norm([p], max_norm=1.0)
print('裁剪后:', p.grad.norm().item())
print('原始范数:', orig)

In [ ]:
# ✅ 提交
from torch_judge import check
check('gradient_clipping')